# Notebook 11: Emissions & Pollution Deep Dive

**One Sensor, One Year — India Breathes**

India's power grid is one of the largest CO2 emitters on Earth. This notebook explores the emissions side of the story:

- Which fuels produce the most CO2?
- Which months are the dirtiest?
- How much CO2 did renewables *prevent*?
- How does grid emissions intensity align with India's worst air quality months?
- What if clean energy didn't exist?

Key numbers from prior analysis:
- Annual CO2: ~1,278 Mt
- Coal share of generation: 73%
- Cleanest day: Aug 26 (587 tCO2/GWh)
- Dirtiest day: Feb 3 (797 tCO2/GWh)

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

df = pd.read_csv('../data/processed/india_2024_derived.csv', parse_dates=['date'])

# Fill NaN with 0 for fuel columns
fuel_cols = ['coal', 'lignite', 'gas', 'nuclear', 'hydro', 'wind', 'solar', 'other_re']
for col in fuel_cols:
    df[col] = df[col].fillna(0)

# Palettes
earth_sky = {
    'coal': '#3D2B1F', 'lignite': '#6B4226', 'gas': '#4A90D9',
    'nuclear': '#2EC4B6', 'hydro': '#1B4F72', 'wind': '#85C1E9',
    'solar': '#F5B041', 'other_re': '#A569BD',
}
co2_color = '#C0392B'
bg_color = '#FAFAF5'

df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%b')

print(f"Data loaded: {len(df)} days, {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Annual CO2: {df['co2_total'].sum()/1e6:.1f} Mt")
print(f"Columns: {list(df.columns)}")

Data loaded: 366 days, 2024-01-01 to 2024-12-31
Annual CO2: 1.3 Mt
Columns: ['date', 'coal', 'lignite', 'gas', 'diesel', 'hsd', 'naphtha', 'multi_fuel', 'thermal', 'nuclear', 'hydro', 'bhutan', 'wind', 'solar', 'other_re', 'renewables', 'total_ex_re', 'total', 'co2_coal', 'co2_lignite', 'co2_gas', 'co2_total', 'emissions_intensity', 're_share', 'clean_share', 'fossil_share', 'wind_solar', 'wind_solar_share', 'month', 'cum_fossil', 'cum_clean', 'cum_total', 'cum_clean_pct', 'month_name']


## 1. Emissions by Source Breakdown

Coal dominates India's CO2 emissions from the power sector. Lignite and gas are minor contributors. Nuclear, hydro, wind, solar, and other renewables produce zero direct CO2.

In [2]:
# Annual CO2 by source — co2 columns are in kt, so /1e3 for Mt
co2_coal_total = df['co2_coal'].sum() / 1e3  # Mt
co2_lignite_total = df['co2_lignite'].sum() / 1e3
co2_gas_total = df['co2_gas'].sum() / 1e3
co2_annual = df['co2_total'].sum() / 1e3

sources = ['Coal', 'Lignite', 'Gas']
values = [co2_coal_total, co2_lignite_total, co2_gas_total]
colors_pie = [earth_sky['coal'], earth_sky['lignite'], earth_sky['gas']]
pcts = [v / co2_annual * 100 for v in values]

print(f"Annual CO2 breakdown:")
for s, v, p in zip(sources, values, pcts):
    print(f"  {s:10s}: {v:6.1f} Mt ({p:.1f}%)")
print(f"  {'Total':10s}: {co2_annual:.1f} Mt")

# Pie chart
fig = go.Figure(go.Pie(
    labels=sources,
    values=values,
    marker=dict(colors=colors_pie),
    textinfo='label+percent',
    hovertemplate='%{label}: %{value:.1f} Mt (%{percent})<extra></extra>',
    hole=0.35,
))
fig.update_layout(
    title=dict(text=f'Power Sector CO2 by Source — {co2_annual:.0f} Mt Total (2024)', font=dict(size=18)),
    height=400, width=500,
    plot_bgcolor=bg_color, paper_bgcolor=bg_color,
)
fig.show()

Annual CO2 breakdown:
  Coal      : 1226.9 Mt (96.0%)
  Lignite   :   37.8 Mt (3.0%)
  Gas       :   13.5 Mt (1.1%)
  Total     : 1278.3 Mt


## 2. Monthly Emissions Profile — Which Months Are Dirtiest?

India's electricity demand peaks in summer (cooling) and winter (heating + low RE). How does CO2 vary month to month?

In [3]:
# Monthly CO2 totals and average emissions intensity
monthly = df.groupby('month').agg(
    co2_total_mt=('co2_total', lambda x: x.sum() / 1e6),
    avg_intensity=('emissions_intensity', 'mean'),
    total_gen=('total', 'sum'),
    co2_coal_mt=('co2_coal', lambda x: x.sum() / 1e6),
    co2_lignite_mt=('co2_lignite', lambda x: x.sum() / 1e6),
    co2_gas_mt=('co2_gas', lambda x: x.sum() / 1e6),
).reset_index()

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly['month_name'] = month_names

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Stacked bar for CO2 by source
fig.add_trace(go.Bar(
    x=monthly['month_name'], y=monthly['co2_coal_mt'],
    name='Coal CO2', marker_color=earth_sky['coal'],
    hovertemplate='Coal: %{y:.1f} Mt<extra></extra>',
), secondary_y=False)

fig.add_trace(go.Bar(
    x=monthly['month_name'], y=monthly['co2_lignite_mt'],
    name='Lignite CO2', marker_color=earth_sky['lignite'],
    hovertemplate='Lignite: %{y:.2f} Mt<extra></extra>',
), secondary_y=False)

fig.add_trace(go.Bar(
    x=monthly['month_name'], y=monthly['co2_gas_mt'],
    name='Gas CO2', marker_color=earth_sky['gas'],
    hovertemplate='Gas: %{y:.2f} Mt<extra></extra>',
), secondary_y=False)

# Line for average emissions intensity
fig.add_trace(go.Scatter(
    x=monthly['month_name'], y=monthly['avg_intensity'],
    name='Avg Intensity (tCO2/GWh)',
    line=dict(color=co2_color, width=3),
    mode='lines+markers',
    hovertemplate='Intensity: %{y:.0f} tCO2/GWh<extra></extra>',
), secondary_y=True)

fig.update_layout(
    barmode='stack',
    title=dict(text='Monthly CO2 Emissions & Emissions Intensity (2024)', font=dict(size=18)),
    plot_bgcolor=bg_color, paper_bgcolor=bg_color,
    height=500,
    legend=dict(orientation='h', y=-0.15),
    xaxis=dict(title=''),
)
fig.update_yaxes(title_text='CO2 Emissions (Mt)', secondary_y=False)
fig.update_yaxes(title_text='Emissions Intensity (tCO2/GWh)', secondary_y=True)

fig.show()

# Identify dirtiest and cleanest months
dirtiest = monthly.loc[monthly['co2_total_mt'].idxmax()]
cleanest = monthly.loc[monthly['co2_total_mt'].idxmin()]
print(f"Dirtiest month: {dirtiest['month_name']} ({dirtiest['co2_total_mt']:.1f} Mt, intensity: {dirtiest['avg_intensity']:.0f} tCO2/GWh)")
print(f"Cleanest month: {cleanest['month_name']} ({cleanest['co2_total_mt']:.1f} Mt, intensity: {cleanest['avg_intensity']:.0f} tCO2/GWh)")

Dirtiest month: May (0.1 Mt, intensity: 721 tCO2/GWh)
Cleanest month: Sep (0.1 Mt, intensity: 644 tCO2/GWh)


## 3. Emissions Avoided by Renewables

**Counterfactual question:** If all clean energy (nuclear + hydro + wind + solar + other RE) had been generated by coal instead, how much *more* CO2 would India have emitted?

We use the coal emission factor: ~950 tCO2 per GWh (Indian coal average, accounting for sub-critical fleet efficiency).

In [4]:
# Coal emission factor — co2_coal is in kt, coal gen is in GWh (MU)
# So coal_ef = kt CO2 per GWh
coal_ef = df['co2_coal'].sum() / df['coal'].sum()
print(f"Implied coal emission factor: {coal_ef:.4f} kt CO2 per GWh = {coal_ef*1000:.0f} tCO2/GWh")

# Clean generation = nuclear + hydro + wind + solar + other_re (in GWh)
clean_cols = ['nuclear', 'hydro', 'wind', 'solar', 'other_re']
df['clean_generation'] = df[clean_cols].sum(axis=1)

# Daily emissions avoided = clean_generation (GWh) * coal_ef (kt/GWh) = kt CO2
df['emissions_avoided'] = df['clean_generation'] * coal_ef  # kt CO2

# Cumulative avoided
df['cum_avoided'] = df['emissions_avoided'].cumsum()

annual_avoided = df['emissions_avoided'].sum()  # kt
co2_annual = df['co2_total'].sum() / 1e3  # Mt
print(f"Annual clean generation: {df['clean_generation'].sum()/1e3:.1f} TWh")
print(f"Annual emissions avoided: {annual_avoided/1e3:.0f} Mt CO2")
print(f"That's {annual_avoided/1e3 / co2_annual * 100:.0f}% of actual emissions")
print(f"Without clean energy, India would emit {annual_avoided/1e3 + co2_annual:.0f} Mt")

# Plot daily avoided emissions
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['date'], y=df['emissions_avoided'],
    fill='tozeroy',
    fillcolor='rgba(46, 196, 182, 0.3)',
    line=dict(color='#2EC4B6', width=1.5),
    name='Daily Avoided CO2',
    hovertemplate='%{x|%b %d}: %{y:.0f} kt CO2 avoided<extra></extra>',
))

fig.add_trace(go.Scatter(
    x=df['date'], y=df['emissions_avoided'].rolling(7).mean(),
    line=dict(color='#1B4F72', width=2.5),
    name='7-day avg',
    hovertemplate='7d avg: %{y:.0f} kt<extra></extra>',
))

fig.update_layout(
    title=dict(text='Daily CO2 Emissions Avoided by Clean Energy (2024)', font=dict(size=18)),
    yaxis_title='CO2 Avoided (kt/day)',
    plot_bgcolor=bg_color, paper_bgcolor=bg_color,
    height=450,
    legend=dict(orientation='h', y=-0.12),
)
fig.show()

# Monthly breakdown of avoided emissions
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_avoided = df.groupby('month').agg(
    avoided_mt=('emissions_avoided', lambda x: x.sum() / 1e3),
).reset_index()
monthly_avoided['month_name'] = month_names

fig2 = go.Figure(go.Bar(
    x=monthly_avoided['month_name'],
    y=monthly_avoided['avoided_mt'],
    marker_color='#2EC4B6',
    hovertemplate='%{x}: %{y:.1f} Mt avoided<extra></extra>',
))
fig2.update_layout(
    title=dict(text='Monthly CO2 Avoided by Clean Energy', font=dict(size=18)),
    yaxis_title='CO2 Avoided (Mt)',
    plot_bgcolor=bg_color, paper_bgcolor=bg_color,
    height=400,
)
fig2.show()

peak_month = monthly_avoided.loc[monthly_avoided['avoided_mt'].idxmax()]
print(f"Peak avoided month: {peak_month['month_name']} ({peak_month['avoided_mt']:.1f} Mt) — monsoon hydro + wind")

Implied coal emission factor: 0.9500 kt CO2 per GWh = 950 tCO2/GWh
Annual clean generation: 410.8 TWh
Annual emissions avoided: 390 Mt CO2
That's 31% of actual emissions
Without clean energy, India would emit 1668 Mt


Peak avoided month: Aug (44.5 Mt) — monsoon hydro + wind


## 4. Seasonal Pollution Parallel — Grid Emissions vs. India's Worst Air Quality

India's worst air quality months are **October through January** — a toxic combination of:
- Post-monsoon crop stubble burning (Oct-Nov)
- Winter temperature inversions trapping pollutants (Nov-Jan)
- Higher fossil fuel use for heating and low solar/wind availability

Does the grid's emissions intensity track this pollution season? The grid contributes to winter pollution through high coal burn and high emissions intensity.

In [5]:
# Emissions intensity with pollution season overlay
fig = go.Figure()

# Daily emissions intensity
fig.add_trace(go.Scatter(
    x=df['date'], y=df['emissions_intensity'],
    mode='lines',
    line=dict(color='#999', width=0.8),
    name='Daily Intensity',
    hovertemplate='%{x|%b %d}: %{y:.0f} tCO2/GWh<extra></extra>',
))

# 7-day rolling average
fig.add_trace(go.Scatter(
    x=df['date'], y=df['emissions_intensity'].rolling(7).mean(),
    mode='lines',
    line=dict(color=co2_color, width=2.5),
    name='7-day avg Intensity',
    hovertemplate='7d avg: %{y:.0f} tCO2/GWh<extra></extra>',
))

# Shade the pollution season (Oct-Jan)
# Oct 1 to Dec 31, 2024
fig.add_shape(
    type='rect',
    x0=pd.Timestamp('2024-10-01'), x1=pd.Timestamp('2024-12-31'),
    y0=0, y1=1, yref='paper',
    fillcolor='rgba(192, 57, 43, 0.12)', line_width=0,
    layer='below',
)
# Jan 1 to Jan 31, 2024 (start of year = end of previous winter)
fig.add_shape(
    type='rect',
    x0=pd.Timestamp('2024-01-01'), x1=pd.Timestamp('2024-01-31'),
    y0=0, y1=1, yref='paper',
    fillcolor='rgba(192, 57, 43, 0.12)', line_width=0,
    layer='below',
)

# Annotations for pollution season
fig.add_annotation(
    x=pd.Timestamp('2024-11-15'), y=0.97, yref='paper',
    text='<b>Pollution Season</b><br>(Oct-Jan: crop burning +<br>winter inversions)',
    showarrow=False, font=dict(size=11, color=co2_color),
    bgcolor='rgba(255,255,255,0.8)',
)
fig.add_annotation(
    x=pd.Timestamp('2024-01-16'), y=0.97, yref='paper',
    text='<b>Winter</b>',
    showarrow=False, font=dict(size=10, color=co2_color),
    bgcolor='rgba(255,255,255,0.8)',
)

# Mark cleanest and dirtiest days
cleanest_day = df.loc[df['emissions_intensity'].idxmin()]
dirtiest_day = df.loc[df['emissions_intensity'].idxmax()]

fig.add_annotation(
    x=pd.Timestamp(cleanest_day['date']),
    y=cleanest_day['emissions_intensity'],
    text=f"Cleanest: {cleanest_day['date'].strftime('%b %d')}<br>{cleanest_day['emissions_intensity']:.0f} tCO2/GWh",
    showarrow=True, arrowhead=2, font=dict(size=10, color='#2EC4B6'),
    ax=0, ay=-40,
)
fig.add_annotation(
    x=pd.Timestamp(dirtiest_day['date']),
    y=dirtiest_day['emissions_intensity'],
    text=f"Dirtiest: {dirtiest_day['date'].strftime('%b %d')}<br>{dirtiest_day['emissions_intensity']:.0f} tCO2/GWh",
    showarrow=True, arrowhead=2, font=dict(size=10, color=co2_color),
    ax=0, ay=-40,
)

fig.update_layout(
    title=dict(text='Grid Emissions Intensity & Pollution Season Overlay (2024)', font=dict(size=18)),
    yaxis_title='Emissions Intensity (tCO2/GWh)',
    plot_bgcolor=bg_color, paper_bgcolor=bg_color,
    height=500,
    legend=dict(orientation='h', y=-0.1),
)
fig.show()

# Stats for pollution vs non-pollution months
pollution_months = [1, 10, 11, 12]
pol = df[df['month'].isin(pollution_months)]
non_pol = df[~df['month'].isin(pollution_months)]
print(f"Pollution season (Oct-Jan) avg intensity: {pol['emissions_intensity'].mean():.0f} tCO2/GWh")
print(f"Non-pollution months avg intensity:       {non_pol['emissions_intensity'].mean():.0f} tCO2/GWh")
print(f"Pollution season avg daily CO2: {pol['co2_total'].mean()/1e3:.0f} kt/day")
print(f"Non-pollution months avg daily CO2: {non_pol['co2_total'].mean()/1e3:.0f} kt/day")
print(f"\nThe grid is {(pol['emissions_intensity'].mean() / non_pol['emissions_intensity'].mean() - 1)*100:.1f}% dirtier during pollution season.")

Pollution season (Oct-Jan) avg intensity: 749 tCO2/GWh
Non-pollution months avg intensity:       707 tCO2/GWh
Pollution season avg daily CO2: 3 kt/day
Non-pollution months avg daily CO2: 4 kt/day

The grid is 5.9% dirtier during pollution season.


## 5. Cumulative CO2 Through the Year — Milestone Markers

When did India's power sector cross 500 Mt and 1,000 Mt of cumulative CO2 in 2024?

In [6]:
# Cumulative CO2 (in Mt) — co2_total is in kt, so /1e3 for Mt
df['cum_co2_mt'] = df['co2_total'].cumsum() / 1e3

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['date'], y=df['cum_co2_mt'],
    fill='tozeroy',
    fillcolor='rgba(192, 57, 43, 0.15)',
    line=dict(color=co2_color, width=2.5),
    name='Cumulative CO2',
    hovertemplate='%{x|%b %d}: %{y:.0f} Mt CO2<extra></extra>',
))

# Find milestone dates
milestones = [500, 1000]
for ms in milestones:
    mask = df['cum_co2_mt'] >= ms
    if mask.any():
        idx = df[mask].index[0]
        ms_date = df.loc[idx, 'date']
        ms_val = df.loc[idx, 'cum_co2_mt']

        fig.add_annotation(
            x=pd.Timestamp(ms_date), y=ms_val,
            text=f"<b>{ms} Mt</b><br>{ms_date.strftime('%b %d')}",
            showarrow=True, arrowhead=2,
            font=dict(size=12, color=co2_color),
            ax=-60, ay=-30,
            bgcolor='rgba(255,255,255,0.8)',
        )
        fig.add_shape(
            type='line',
            x0=pd.Timestamp(ms_date), x1=pd.Timestamp(ms_date),
            y0=0, y1=ms_val,
            line=dict(color=co2_color, width=1, dash='dot'),
        )
        print(f"India crossed {ms} Mt on {ms_date.strftime('%B %d, %Y')} (day {(ms_date - pd.Timestamp('2024-01-01')).days + 1} of 366)")

# Final total annotation
final = df.iloc[-1]
fig.add_annotation(
    x=pd.Timestamp(final['date']), y=final['cum_co2_mt'],
    text=f"<b>Total: {final['cum_co2_mt']:.0f} Mt</b>",
    showarrow=True, arrowhead=2,
    font=dict(size=13, color=co2_color),
    ax=-70, ay=-20,
    bgcolor='rgba(255,255,255,0.8)',
)

fig.update_layout(
    title=dict(text='Cumulative CO2 Emissions from India\'s Power Grid (2024)', font=dict(size=18)),
    yaxis_title='Cumulative CO2 (Mt)',
    plot_bgcolor=bg_color, paper_bgcolor=bg_color,
    height=500,
    showlegend=False,
)
fig.show()

# Average rate
avg_daily = df['co2_total'].mean()  # kt/day
print(f"\nAverage rate: {avg_daily:.0f} kt CO2/day = {avg_daily*365/1e3:.0f} Mt/year pace")
print(f"Final total: {final['cum_co2_mt']:.1f} Mt")

India crossed 500 Mt on May 16, 2024 (day 137 of 366)
India crossed 1000 Mt on October 09, 2024 (day 283 of 366)



Average rate: 3493 kt CO2/day = 1275 Mt/year pace
Final total: 1278.3 Mt


## 6. "What If" Scenario — India Without Clean Energy

If India's 2024 clean energy generation (nuclear, hydro, wind, solar, other RE) didn't exist and was replaced entirely by coal, what would total emissions look like? This is a stark counterfactual that shows how much worse things could be.

In [7]:
# "What if" — all clean replaced by coal
# emissions_avoided is in kt CO2, co2_total is in kt CO2
df['co2_counterfactual'] = df['co2_total'] + df['emissions_avoided']
df['cum_co2_actual'] = df['co2_total'].cumsum() / 1e3  # kt → Mt
df['cum_co2_counterfactual'] = df['co2_counterfactual'].cumsum() / 1e3

fig = go.Figure()

# Counterfactual (all coal)
fig.add_trace(go.Scatter(
    x=df['date'], y=df['cum_co2_counterfactual'],
    fill='tozeroy',
    fillcolor='rgba(192, 57, 43, 0.1)',
    line=dict(color=co2_color, width=2, dash='dash'),
    name='Without Clean Energy',
    hovertemplate='No clean: %{y:.0f} Mt<extra></extra>',
))

# Actual
fig.add_trace(go.Scatter(
    x=df['date'], y=df['cum_co2_actual'],
    fill='tozeroy',
    fillcolor='rgba(46, 196, 182, 0.2)',
    line=dict(color='#2EC4B6', width=2.5),
    name='Actual Emissions',
    hovertemplate='Actual: %{y:.0f} Mt<extra></extra>',
))

# Annotation for the gap
gap_final = df['cum_co2_counterfactual'].iloc[-1] - df['cum_co2_actual'].iloc[-1]
mid_date = pd.Timestamp('2024-07-01')
mid_idx = df[df['date'] >= mid_date].index[0]
mid_actual = df.loc[mid_idx, 'cum_co2_actual']
mid_cf = df.loc[mid_idx, 'cum_co2_counterfactual']

fig.add_annotation(
    x=pd.Timestamp('2024-07-01'), y=(mid_actual + mid_cf) / 2,
    text=f'<b>Emissions avoided<br>by clean energy:<br>{gap_final:.0f} Mt CO2</b>',
    showarrow=False,
    font=dict(size=13, color='#1B4F72'),
    bgcolor='rgba(255,255,255,0.85)',
)

# End labels
fig.add_annotation(
    x=pd.Timestamp(df['date'].iloc[-1]),
    y=df['cum_co2_counterfactual'].iloc[-1],
    text=f"<b>{df['cum_co2_counterfactual'].iloc[-1]:.0f} Mt</b><br>(all coal)",
    showarrow=True, arrowhead=2, ax=-60, ay=-10,
    font=dict(size=11, color=co2_color),
    bgcolor='rgba(255,255,255,0.8)',
)
fig.add_annotation(
    x=pd.Timestamp(df['date'].iloc[-1]),
    y=df['cum_co2_actual'].iloc[-1],
    text=f"<b>{df['cum_co2_actual'].iloc[-1]:.0f} Mt</b><br>(actual)",
    showarrow=True, arrowhead=2, ax=-60, ay=20,
    font=dict(size=11, color='#2EC4B6'),
    bgcolor='rgba(255,255,255,0.8)',
)

fig.update_layout(
    title=dict(text='What If India Had No Clean Energy? Cumulative CO2 Comparison (2024)', font=dict(size=17)),
    yaxis_title='Cumulative CO2 (Mt)',
    plot_bgcolor=bg_color, paper_bgcolor=bg_color,
    height=550,
    legend=dict(orientation='h', y=-0.1),
)
fig.show()

pct_increase = (df['cum_co2_counterfactual'].iloc[-1] / df['cum_co2_actual'].iloc[-1] - 1) * 100
print(f"Actual 2024 CO2: {df['cum_co2_actual'].iloc[-1]:.0f} Mt")
print(f"Counterfactual (no clean): {df['cum_co2_counterfactual'].iloc[-1]:.0f} Mt")
print(f"Emissions avoided by clean energy: {gap_final:.0f} Mt ({pct_increase:.0f}% increase without it)")
print(f"\nWithout clean energy, India's power sector emissions would be {pct_increase:.0f}% higher.")

Actual 2024 CO2: 1278 Mt
Counterfactual (no clean): 1668 Mt
Emissions avoided by clean energy: 390 Mt (31% increase without it)

Without clean energy, India's power sector emissions would be 31% higher.


## 7. Key Findings

### Emissions by Source
- **Coal produces ~96% of India's power sector CO2**, lignite ~3%, gas ~1%.
- Coal is not just the dominant fuel (73% of generation) — it is the *near-exclusive* source of carbon.

### Monthly Pattern
- **Winter months (Oct-Feb) are the dirtiest** — high demand, low RE availability, and high emissions intensity converge.
- **Monsoon months (Jul-Sep) are the cleanest** — hydro surges, wind is strong, and coal retreats.

### Emissions Avoided
- Clean energy (nuclear + hydro + wind + solar + other RE) **prevented hundreds of Mt of CO2** that would have been emitted if coal had filled the gap.
- Avoided emissions peak during monsoon when hydro and wind are strongest.

### Pollution Season Parallel
- India's worst air quality months (Oct-Jan) coincide with the grid's highest emissions intensity period.
- The grid is measurably dirtier during pollution season — higher coal share, lower RE share, winter inversions trap the pollutants.
- **The grid doesn't just correlate with winter pollution — it actively contributes to it.**

### The "No Clean Energy" Counterfactual
- Without clean energy, India's power sector CO2 would be dramatically higher.
- Clean energy is already making a significant dent — but coal's dominance means the absolute emissions remain enormous.

### Milestones
- India's power sector crossed **500 Mt** of cumulative CO2 in roughly the first 5 months of 2024.
- It crossed **1,000 Mt** around September-October.
- The year ended at approximately **1,278 Mt** — all from a single sector of a single country.